# 📅 Seasonal Patterns Analysis

Discover when to book flights for the best prices and analyze trends over time.

## Setup

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_excel('airline_ticket_dataset.xlsx')
print(f"✅ Loaded {len(df):,} flight records from {df['Year'].min()} to {df['Year'].max()}")

## 1. Fare Trends Over Time (by Year and Quarter)

How have prices changed?

In [ ]:
# Create time period column
df['period'] = df['Year'].astype(str) + ' Q' + df['quarter'].astype(str)

# Calculate average fare by period
time_series = df.groupby(['Year', 'quarter', 'period']).agg({
    'fare': 'mean',
    'passengers': 'sum'
}).reset_index()

time_series = time_series.sort_values(['Year', 'quarter'])

fig = px.line(time_series, 
              x='period', 
              y='fare',
              title='Average Fare Trends Over Time',
              labels={'period': 'Year-Quarter', 'fare': 'Average Fare ($)'},
              markers=True)

fig.update_traces(line_color='#636EFA', line_width=3, marker=dict(size=10))
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

## 2. Quarterly Fare Comparison

Which quarter is cheapest to fly?

In [ ]:
quarter_stats = df.groupby('quarter').agg({
    'fare': 'mean',
    'passengers': 'sum'
}).reset_index()

quarter_labels = {1: 'Q1 (Jan-Mar)', 2: 'Q2 (Apr-Jun)', 3: 'Q3 (Jul-Sep)', 4: 'Q4 (Oct-Dec)'}
quarter_stats['Quarter Label'] = quarter_stats['quarter'].map(quarter_labels)

fig = px.bar(quarter_stats, 
             x='Quarter Label', 
             y='fare',
             title='Average Fare by Quarter (Across All Years)',
             labels={'fare': 'Average Fare ($)', 'Quarter Label': 'Quarter'},
             color='fare',
             color_continuous_scale='RdYlGn_r',
             text='fare')

fig.update_traces(texttemplate='$%{text:.2f}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

# Find cheapest quarter
cheapest_q = quarter_stats.loc[quarter_stats['fare'].idxmin()]
print(f"\n💰 Cheapest quarter to fly: {cheapest_q['Quarter Label']} (avg ${cheapest_q['fare']:.2f})")

## 3. Passenger Volume by Quarter

When do most people travel?

In [ ]:
fig = px.bar(quarter_stats, 
             x='Quarter Label', 
             y='passengers',
             title='Total Passenger Volume by Quarter',
             labels={'passengers': 'Total Passengers', 'Quarter Label': 'Quarter'},
             color='passengers',
             color_continuous_scale='Blues',
             text='passengers')

fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

## 4. Yearly Comparison

How do years compare?

In [ ]:
yearly_stats = df.groupby('Year').agg({
    'fare': 'mean',
    'passengers': 'sum'
}).reset_index()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=yearly_stats['Year'],
    y=yearly_stats['fare'],
    name='Average Fare',
    marker_color='indianred',
    text=yearly_stats['fare'],
    texttemplate='$%{text:.2f}',
    textposition='outside'
))

fig.update_layout(
    title='Average Fare by Year',
    xaxis_title='Year',
    yaxis_title='Average Fare ($)',
    showlegend=False
)
fig.show()

## 5. Seasonal Demand Heatmap

Visualize fare patterns across years and quarters:

In [ ]:
# Create pivot table for heatmap
heatmap_data = time_series.pivot(index='Year', columns='quarter', values='fare')

fig = px.imshow(heatmap_data,
                labels=dict(x='Quarter', y='Year', color='Avg Fare ($)'),
                title='Fare Heatmap: Years vs Quarters',
                color_continuous_scale='RdYlGn_r',
                aspect='auto',
                text_auto='.2f')

fig.update_xaxes(side='top')
fig.show()

## 6. Quarter-over-Quarter Change

How much do fares change from one quarter to the next?

In [ ]:
# Calculate QoQ change
time_series['fare_change'] = time_series['fare'].pct_change() * 100

fig = px.bar(time_series[1:],  # Skip first row (no previous quarter)
             x='period', 
             y='fare_change',
             title='Quarter-over-Quarter Fare Change (%)',
             labels={'period': 'Year-Quarter', 'fare_change': 'Change from Previous Quarter (%)'},
             color='fare_change',
             color_continuous_scale='RdYlGn',
             color_continuous_midpoint=0)

fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

## 7. Best Time to Book: Quarter and Carrier Analysis

Find the cheapest quarter by carrier:

In [ ]:
# Top 5 carriers by passenger volume
top_carriers = df.groupby('carrier_lg')['passengers'].sum().nlargest(5).index

carrier_quarterly = df[df['carrier_lg'].isin(top_carriers)].groupby(['carrier_lg', 'quarter']).agg({
    'fare': 'mean'
}).reset_index()

carrier_quarterly['Quarter'] = carrier_quarterly['quarter'].map(quarter_labels)

fig = px.line(carrier_quarterly, 
              x='Quarter', 
              y='fare',
              color='carrier_lg',
              title='Seasonal Fare Patterns by Top 5 Carriers',
              labels={'fare': 'Average Fare ($)', 'carrier_lg': 'Carrier'},
              markers=True)

fig.update_layout(height=500)
fig.show()

## 8. Passenger Trends Over Time

In [ ]:
fig = px.area(time_series, 
              x='period', 
              y='passengers',
              title='Passenger Volume Trends Over Time',
              labels={'period': 'Year-Quarter', 'passengers': 'Total Passengers'})

fig.update_traces(line_color='#00CC96', fillcolor='rgba(0, 204, 150, 0.3)')
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

## 💡 Seasonal Insights Summary

In [ ]:
print("="*60)
print("📅 SEASONAL PATTERNS INSIGHTS")
print("="*60)

# Best time to book
cheapest_q = quarter_stats.loc[quarter_stats['fare'].idxmin()]
most_exp_q = quarter_stats.loc[quarter_stats['fare'].idxmax()]

print(f"\n💰 Best Time to Book:")
print(f"   • Cheapest quarter: {cheapest_q['Quarter Label']} (${cheapest_q['fare']:.2f})")
print(f"   • Most expensive: {most_exp_q['Quarter Label']} (${most_exp_q['fare']:.2f})")
print(f"   • Potential savings: ${most_exp_q['fare'] - cheapest_q['fare']:.2f}")

# Busiest period
busiest_q = quarter_stats.loc[quarter_stats['passengers'].idxmax()]
print(f"\n✈️ Busiest Travel Period:")
print(f"   • Quarter: {busiest_q['Quarter Label']}")
print(f"   • Total passengers: {busiest_q['passengers']:,.0f}")

# Yearly trend
first_year = yearly_stats.iloc[0]
last_year = yearly_stats.iloc[-1]
fare_change = last_year['fare'] - first_year['fare']
fare_change_pct = (fare_change / first_year['fare']) * 100

print(f"\n📈 Overall Trend ({first_year['Year']} to {last_year['Year']}):")
print(f"   • Fare change: ${fare_change:.2f} ({fare_change_pct:+.1f}%)")
print(f"   • {first_year['Year']} avg: ${first_year['fare']:.2f}")
print(f"   • {last_year['Year']} avg: ${last_year['fare']:.2f}")

# Volatility
fare_volatility = time_series['fare'].std()
print(f"\n📊 Price Volatility:")
print(f"   • Standard deviation: ${fare_volatility:.2f}")

print("\n" + "="*60)